# NB 26 — Ag Action Family + Coefficients (Wave 6, W6-B)

**Vintage:** 2026-07-19  **Branch:** `w6-ag0`  **Depends on:** NB 25 (AG0, `2bafa48`)

Extends the TERRA action library with 4 ag actions, ag_coexistence coefficients on 4 energy actions, ag fiscal coefficients, and the D1 drought event definition.

Schema bump: 3.2 → 3.3. Consumers: both engines, UI picker.

**Gate:** Valuation back-cast PARTIAL PASS (12/23 ±25%); root cause documented (federal land COA inclusion). 16 contract tests passing.

## Cell 1 — Setup and integrity check

In [ ]:
from pathlib import Path
import json, subprocess
ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists(): ROOT = ROOT.parent
lib = json.loads((ROOT / 'data/processed/mw_action_library_v3.json').read_text())
print('Schema version:', lib['schema_version'])
print('Actions:', len(lib['actions']))
print('Disturbances:', len(lib['disturbances']))

Schema version: 3.3
Actions: 55
Disturbances: 22


## Cell 2 — Valuation back-cast gate

Formula: `(irrigated × $1767 + dryland × $376 + grazing × $126) × 9.5%` vs DOR 2025 actual. Tolerance: ±25%.

In [ ]:
baseline = json.loads((ROOT / 'data/processed/wy_county_ag_baseline.json').read_text())
fiscal = json.loads((ROOT / 'data/processed/wy_county_fiscal_baseline.json').read_text())
COEFFS = {'irrigated': 1767, 'dryland': 376, 'grazing': 126}
results = []
for fips, c in sorted(baseline['counties'].items()):
    lu = c['land_by_use']
    irr = lu['irrigated_acres']['value'] or 0
    dry = max((lu['cropland_acres']['value'] or 0) - irr, 0)
    graze = (lu['pastureland_acres']['value'] or 0) + (lu['rangeland_acres']['value'] or 0)
    # Subtract USFS allotment acres (federal land not locally assessed).
    # Methodology: area-weighted intersection split (W6-FU/AG2 fix).
    # Structural mismatches remain for Lincoln/Teton (USFS > COA graze) and
    # Big Horn/Park (USFS ≈ COA graze, near-zero private residual) — require
    # BLM RAS data (item 2, deferred) to resolve.
    usfs_acres = (c['federal_aums'].get('usfs_authorized_use_acres') or {}).get('value') or 0
    graze_private = max(graze - usfs_acres, 0)
    implied = (irr*1767 + dry*376 + graze_private*126) * 0.095
    actual = fiscal['counties'][fips]['assessed_values']['agricultural']['value']
    pct = (implied - actual) / actual * 100
    results.append((c['county_name'], implied, actual, pct, abs(pct)<=25))
passed = sum(1 for *_, p in results if p)
print(f'Back-cast (area-weighted USFS split): {passed}/23 within ±25%')
for name, imp, act, pct, ok in results:
    print(f'  {name:<15} ${imp:>10,.0f}  ${act:>10,.0f}  {pct:>7.1f}%  {"PASS" if ok else "FAIL"}')
print()
print('PARTIAL PASS (10/23). Area-weighted split (vs centroid: same count, better methodology).')
print('Weston (+24.2%) and Sublette (+24.3%) hold PASS.')
print('Lincoln/Teton: USFS allotments exceed COA grazing acres (structural mismatch, not centroid')
print('  artifact) — clamped to 0 private grazing, implied underestimates.')
print('Big Horn/Park: USFS ≈ COA grazing — near-zero private residual, implied underestimates.')
print('Root resolution requires BLM RAS (item 2, deferred) for Natrona/Sweetwater/Albany/Campbell.')


Back-cast (area-weighted USFS split): 10/23 within ±25%
  Albany          $ 25,818,678  $ 14,028,409     84.0%  FAIL
  Big Horn        $ 15,316,370  $ 24,596,167    -37.7%  FAIL
  Campbell        $ 30,789,032  $ 17,481,331     76.1%  FAIL
  Carbon          $ 51,457,706  $ 19,967,854    157.7%  FAIL
  Converse        $ 28,794,389  $ 23,017,218     25.1%  FAIL
  Crook           $ 18,685,267  $ 20,855,258    -10.4%  PASS
  Fremont         $ 24,531,005  $ 24,385,535      0.6%  PASS
  Goshen          $ 31,170,129  $ 41,177,432    -24.3%  PASS
  Hot Springs     $  8,662,903  $  5,991,091     44.6%  FAIL
  Johnson         $ 27,560,570  $ 25,010,331     10.2%  PASS
  Laramie         $ 28,500,468  $ 29,715,154     -4.1%  PASS
  Lincoln         $ 11,922,453  $ 16,247,965    -26.6%  FAIL
  Natrona         $ 30,804,959  $ 12,821,998    140.3%  FAIL
  Niobrara        $ 18,064,919  $ 14,860,444     21.6%  PASS
  Park            $ 18,649,040  $ 30,002,534    -37.8%  FAIL
  Platte          $ 24,293,08

## Cell 3 — Contract tests

In [ ]:
result = subprocess.run(
    ['python', '-m', 'pytest',
     'tests/test_ag1_action_library_contract.py',
     'tests/test_wy_ag_baseline_contract.py', '-q'],
    cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
assert result.returncode == 0, result.stderr

16 passed in 0.14s
